In [2]:
pip install pm4py

Defaulting to user installation because normal site-packages is not writeable
  Using cached pm4py-2.7.23.8-py3-none-any.whl.metadata (8.0 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
Using cached pm4py-2.7.23.8-py3-none-any.whl (2.7 MB)
   ---------------------------------------- 0.0/13.6 MB ? eta -:--:--
   ----------------------- ---------------- 7.9/13.6 MB 44.2 MB/s eta 0:00:01
   ---------------------------------------- 13.6/13.6 MB 42.5 MB/s eta 0:00:00
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pm4py



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing


C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\reshu\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
import pandas as pd

In [3]:
import time

In [5]:
def xes_to_csv_pipeline(xes_path: str, csv_output_path: str):
    print(f"[+] Initializing parsing for: {xes_path}")
    start_time = time.time()
    
    # Parse XES XML log into PM4Py Event Log object
    log = pm4py.read_xes(xes_path)
    
    # Convert hierarchical log object into flat Pandas DataFrame
    df = pm4py.convert_to_dataframe(log)
    
    print(f"[+] Raw extracted columns ({len(df.columns)} total):")
    print(df.columns.tolist())
    
    # Map BPI 2019 XES attribute keys to standardized analytical column names
    column_mapping = {
        'case:concept:name': 'case_id',
        'concept:name': 'activity_name',
        'time:timestamp': 'event_timestamp',
        'org:resource': 'resource_id',
        'case:RequestedAmount': 'net_amount'
    }
    
    df = df.rename(columns=column_mapping)
    
    # Retain core process mining columns
    keep_cols = ['case_id', 'activity_name', 'event_timestamp', 'resource_id']
    if 'net_amount' in df.columns:
        keep_cols.append('net_amount')
        
    df_clean = df[keep_cols].copy()
    
    # Parse timestamps to standard ISO format string
    df_clean['event_timestamp'] = pd.to_datetime(df_clean['event_timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
    df_clean['resource_id'] = df_clean['resource_id'].fillna('SYSTEM_AUTOMATED')
    
    # Sort chronologically per case to enforce sequence order
    df_clean = df_clean.sort_values(by=['case_id', 'event_timestamp']).reset_index(drop=True)
    
    # Export to disk
    df_clean.to_csv(csv_output_path, index=False)
    
    print(f"[+] Exported {len(df_clean):,} records to '{csv_output_path}' in {time.time() - start_time:.2f} seconds.")

if __name__ == "__main__":
    xes_to_csv_pipeline("BPI_Challenge_2019.xes", "bpi_2019_raw.csv")

[+] Initializing parsing for: BPI_Challenge_2019.xes


parsing log, completed traces ::   0%|          | 0/251734 [00:00<?, ?it/s]

[+] Raw extracted columns (21 total):
['User', 'org:resource', 'concept:name', 'Cumulative net worth (EUR)', 'time:timestamp', 'case:Spend area text', 'case:Company', 'case:Document Type', 'case:Sub spend area text', 'case:Purchasing Document', 'case:Purch. Doc. Category name', 'case:Vendor', 'case:Item Type', 'case:Item Category', 'case:Spend classification text', 'case:Source', 'case:Name', 'case:GR-Based Inv. Verif.', 'case:Item', 'case:concept:name', 'case:Goods Receipt']
[+] Exported 1,595,923 records to 'bpi_2019_raw.csv' in 112.59 seconds.


In [6]:
import numpy as np

In [7]:
import pandas as pd

In [14]:
# Load row dataset
df = pd.read_csv("bpi_2019_raw.csv")

# 1. Structural Overview & Memory Footprint
print("=== DATASET DIMENSIONS ===")
print(f"Total Rows (Events): {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print("\n=== DATA TYPES & NULL COUNT ===")
print(df.info())

print("\n=== FIRST 10 ROWS (HEAD) ===")
display(df.head(10))

print("\n=== RANDOM SAMPLE (10 ROWS) ====")
display(df.sample(10, random_state=42))

print("\n=== CARDINALITY (UNIQUE VALUES PER COLUMN) ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique():,} unique values")
        

=== DATASET DIMENSIONS ===
Total Rows (Events): 1,595,923
Total Columns: 4
Memory Usage: 145.76 MB

=== DATA TYPES & NULL COUNT ===
<class 'pandas.DataFrame'>
RangeIndex: 1595923 entries, 0 to 1595922
Data columns (total 4 columns):
 #   Column           Non-Null Count    Dtype
---  ------           --------------    -----
 0   case_id          1595923 non-null  str  
 1   activity_name    1595923 non-null  str  
 2   event_timestamp  1595923 non-null  str  
 3   resource_id      1595923 non-null  str  
dtypes: str(4)
memory usage: 145.8 MB
None

=== FIRST 10 ROWS (HEAD) ===


,case_id,activity_name,event_timestamp,resource_id
0,2000000000_00001,SRM: Created,2018-01-02 12:53:00,batch_00
1,2000000000_00001,SRM: Complete,2018-01-02 13:53:00,batch_00
2,2000000000_00001,SRM: Awaiting Approval,2018-01-02 13:53:00,batch_00
3,2000000000_00001,SRM: Document Completed,2018-01-02 13:53:00,batch_00
4,2000000000_00001,SRM: In Transfer to Execution Syst.,2018-01-02 13:53:00,batch_00
5,2000000000_00001,SRM: Ordered,2018-01-02 13:53:00,batch_00
6,2000000000_00001,SRM: Change was Transmitted,2018-01-02 13:53:00,batch_00
7,2000000000_00001,Create Purchase Order Item,2018-01-02 13:53:00,user_000
8,2000000000_00001,Vendor creates invoice,2018-01-02 22:59:00,NONE
9,2000000000_00001,Record Goods Receipt,2018-03-06 06:44:00,user_000



=== RANDOM SAMPLE (10 ROWS) ====


,case_id,activity_name,event_timestamp,resource_id
427192,4507018924_00010,Receive Order Confirmation,2018-03-29 13:50:00,user_029
697067,4507030677_00010,Create Purchase Order Item,2018-05-22 13:12:00,user_158
869342,4507039206_00010,Record Invoice Receipt,2018-07-25 09:30:00,user_013
120044,4507004349_00140,Remove Payment Block,2018-01-29 16:11:00,user_006
1191344,4508054573_00010,Clear Invoice,2019-01-04 22:59:00,NONE
39688,4507000879_00020,Vendor creates invoice,2018-05-02 21:59:00,NONE
1422970,4508066069_00010,Record Goods Receipt,2018-11-29 12:00:00,user_088
1494046,4508069901_00020,Record Goods Receipt,2018-12-04 12:04:00,user_053
1340037,4508061832_00010,Create Purchase Order Item,2018-10-16 08:45:00,user_178
335739,4507014785_00010,Record Goods Receipt,2018-03-19 12:36:00,user_136



=== CARDINALITY (UNIQUE VALUES PER COLUMN) ===
case_id: 251,734 unique values
activity_name: 42 unique values
event_timestamp: 167,432 unique values
resource_id: 628 unique values


In [15]:
# Count how many rows were assigned the automated placeholder
automated_count = (df['resource_id'] == 'SYSTEM_AUTOMATED').sum()
print(f"Total automated system events: {automated_count:,}")

Total automated system events: 0


In [16]:
# Group by case_id and calculate total events vs unique activities
case_stats = df.groupby('case_id')['activity_name'].agg(
    total_events='count',
    unique_activities='nunique'
)

# Identify cases where total event steps exceed unique activity steps
rework_cases = case_stats[case_stats['total_events'] > case_stats['unique_activities']]
print(f"Cases with Rework Loops: {len(rework_cases):,}")

Cases with Rework Loops: 22,438


In [17]:
# Calculate activity frequency and cumulative percentage
activity_counts = df['activity_name'].value_counts().reset_index()
activity_counts.columns = ['activity_name', 'count']
activity_counts['percentage'] = (activity_counts['count'] / len(df)) * 100
activity_counts['cumulative_percentage'] = activity_counts['percentage'].cumsum()

display(activity_counts.head(10))

,activity_name,count,percentage,cumulative_percentage
0,Record Goods Receipt,314097,19.681213,19.681213
1,Create Purchase Order Item,251734,15.773568,35.454781
2,Record Invoice Receipt,228760,14.334025,49.788806
3,Vendor creates invoice,219919,13.780051,63.568856
4,Clear Invoice,194393,12.180600,75.749457
5,Record Service Entry Sheet,164975,10.337278,86.086735
6,Remove Payment Block,57136,3.580123,89.666857
7,Create Purchase Requisition Item,46592,2.919439,92.586296
8,Receive Order Confirmation,32065,2.009182,94.595479
9,Change Quantity,21449,1.343987,95.939466


In [18]:
# Compute statistical threshold for case step count
events_per_case = df.groupby('case_id')['activity_name'].count()

q1 = events_per_case.quantile(0.25)
q3 = events_per_case.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + (1.5 * iqr)

print(f"Mean Steps: {events_per_case.mean():.2f}")
print(f"75th Percentile (Q3): {q3}")
print(f"Statistical Outlier Threshold (Q3 + 1.5*IQR): {upper_bound:.1f} steps")

Mean Steps: 6.34
75th Percentile (Q3): 6.0
Statistical Outlier Threshold (Q3 + 1.5*IQR): 7.5 steps


In [19]:
pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 31.7 MB/s eta 0:00:00
